In [ ]:
import networkx as nx
import pandas as pd
import numpy as np
import random
import pickle
from scipy.spatial import KDTree
import os

def fragility(node_type, depth):
    if node_type == 'road':
        return 1 / (1 + np.exp(-10 * (depth - 0.3)))
    elif node_type in ('substation', 'power'):
        return 1 / (1 + np.exp(-6 * (depth - 1.2)))
    elif node_type == 'hospital':
        return 1 / (1 + np.exp(-4 * (depth - 1.5)))
    return 0.0

def build_dependency_map(graph):
    substations = [n for n, d in graph.nodes(data=True) if d.get('node_type') in ('substation', 'power')]
    roads = [n for n, d in graph.nodes(data=True) if d.get('node_type') == 'road']
    hospitals = [n for n, d in graph.nodes(data=True) if d.get('node_type') == 'hospital']
    
    def get_coords(nodes):
        coords = []
        valid_nodes = []
        for n in nodes:
            d = graph.nodes[n]
            if 'x' in d and 'y' in d:
                coords.append([d['x'], d['y']])
                valid_nodes.append(n)
        return valid_nodes, np.array(coords)
        
    sub_nodes, sub_coords = get_coords(substations)
    road_nodes, road_coords = get_coords(roads)
    
    sub_tree = KDTree(sub_coords) if len(sub_coords) > 0 else None
    road_tree = KDTree(road_coords) if len(road_coords) > 0 else None
    
    dependencies = {}
    for n in graph.nodes():
        d = graph.nodes[n]
        node_type = d.get('node_type', 'road')
        deps = {}
        if node_type == 'hospital':
            if sub_tree and 'x' in d and 'y' in d:
                _, idx = sub_tree.query([d['x'], d['y']])
                deps['nearest_substation'] = sub_nodes[idx]
            if road_tree and 'x' in d and 'y' in d:
                _, idx = road_tree.query([d['x'], d['y']])
                deps['nearest_road'] = road_nodes[idx]
        elif node_type in ('substation', 'power'):
            if road_tree and 'x' in d and 'y' in d:
                _, idx = road_tree.query([d['x'], d['y']])
                deps['nearest_road'] = road_nodes[idx]
        dependencies[n] = deps
    return dependencies

def compute_dependency_stress(node, node_type, dependencies, failed_nodes):
    deps = dependencies.get(node, {})
    if node_type == 'hospital':
        sub_failed = int(deps.get('nearest_substation') in failed_nodes)
        road_failed = int(deps.get('nearest_road') in failed_nodes)
        return 0.6 * sub_failed + 0.4 * road_failed
    elif node_type in ('substation', 'power'):
        road_failed = int(deps.get('nearest_road') in failed_nodes)
        return float(road_failed)
    return 0.0

def redistribute_load(graph, failed_nodes, newly_failed_nodes, betweenness, capacity, current_load):
    redistributed_load = {n: 0.0 for n in graph.nodes()}
    
    for node in graph.nodes():
        if node in failed_nodes:
            continue
        degree = graph.degree(node)
        if degree == 0:
            continue
            
        newly_failed_neighbors = [ng for ng in list(graph.neighbors(node)) if ng in newly_failed_nodes]
        load = sum((betweenness[fn] / degree) for fn in newly_failed_neighbors)
        redistributed_load[node] = load
        
    overload_ratio = {}
    for n in graph.nodes():
        if n not in failed_nodes:
            current_load[n] += redistributed_load[n]
        if n in newly_failed_nodes:
            current_load[n] = 0.0
            
        if capacity[n] > 0:
            overload_ratio[n] = current_load[n] / capacity[n]
        else:
            overload_ratio[n] = 0.0
            
    return redistributed_load, overload_ratio

def compute_failure_probability(fragility_score, overload_ratio, dependency_stress):
    # P(failure) = w1 * fragility_score + w2 * overload_effect + w3 * dependency_stress
    w1, w2, w3 = 0.5, 0.2, 0.3
    overload_effect = 1 - np.exp(-overload_ratio)
    # Improve failure probability formulation
    p = 1 - np.exp(-(w1 * fragility_score + w2 * overload_effect + w3 * dependency_stress))
    return min(1.0, max(0.0, p))

def simulate_cascade(graph, scenario_id, n_timesteps, dependencies, betweenness, capacity, base_flood_depths, degree_cent):
    records = []
    failed_nodes = set()
    newly_failed_nodes = set()
    current_load = {n: betweenness[n] for n in graph.nodes()}
    
    for t in range(n_timesteps):
        redistributed_load, overload_ratio = redistribute_load(graph, failed_nodes, newly_failed_nodes, betweenness, capacity, current_load)
        new_failures = set()
        
        for node, attr in graph.nodes(data=True):
            node_type = attr.get('node_type', 'road')
            elevation = attr.get('elevation_m', 0.0)
            dist_epi = attr.get('distance_from_epicenter', 0.0)
            
            was_failed_prev = int(node in failed_nodes)
            neighbors = list(graph.neighbors(node))
            failed_neighbors_prev = sum(1 for ng in neighbors if ng in failed_nodes)
            
            if was_failed_prev:
                failed = 1
                depth = base_flood_depths[node]
                fragility_score = 1.0
                dep_stress = 1.0
                overload = overload_ratio.get(node, 0.0)
            else:
                all_distances = [d.get('distance_from_epicenter', 0.0) for n, d in graph.nodes(data=True)]
                scale = np.percentile(all_distances, 50) if all_distances else 0.5
                if scale == 0:
                    scale = 0.5
                scenario_multiplier = 1.0 + (0.1 * scenario_id)
                depth_decay = np.exp(-dist_epi / scale) if dist_epi > 0 else 1.0
                depth = base_flood_depths[node] * scenario_multiplier * depth_decay
                depth += max(0.0, np.random.normal(0, 0.05)) if scenario_id > 0 else 0.0
                
                fragility_score = fragility(node_type, depth)
                
                dep_stress = compute_dependency_stress(node, node_type, dependencies, failed_nodes)
                dep_stress += 0.1 * failed_neighbors_prev
                dep_stress = min(dep_stress, 1.0)
                
                overload = overload_ratio.get(node, 0.0)
                p_failure = compute_failure_probability(fragility_score, overload, dep_stress)
                
                if overload > 1.5:
                    p_failure = 1.0
                    
                failed = int(np.random.binomial(1, p_failure))
            
            records.append({
                'scenario_id': scenario_id,
                'timestep': t,
                'node_id': node,
                'node_type': node_type,
                'elevation': elevation,
                'flood_depth': depth,
                'fragility_score': fragility_score,
                'degree_centrality': degree_cent.get(node, 0.0),
                'betweenness_centrality': betweenness.get(node, 0.0),
                'redistributed_load': redistributed_load.get(node, 0.0),
                'overload_ratio': overload,
                'failed_neighbors_prev': failed_neighbors_prev,
                'cross_layer_dependency_stress': dep_stress,
                'distance_from_epicenter': dist_epi,
                'was_failed_prev': was_failed_prev,
                'target_failed': failed
            })
            
            if failed and not was_failed_prev:
                new_failures.add(node)
                
        failed_nodes.update(new_failures)
        newly_failed_nodes = new_failures
        
    return records


def generate_synthetic_dataset(graph, n_scenarios=2, n_timesteps=5, seed=42):
    np.random.seed(seed)
    random.seed(seed)
    
    print("Building dependency map...")
    dependencies = build_dependency_map(graph)
    
    print("Computing centralities...")
    degree_cent = nx.degree_centrality(graph)
    
    if len(graph) > 5000:
        print("Graph is large, computing approximated betweenness centrality...")
        betweenness = nx.betweenness_centrality(graph, k=min(100, len(graph)), seed=seed)
    else:
        betweenness = nx.betweenness_centrality(graph)
        
    alpha = 0.2
    capacity = {}
    for n in graph.nodes():
        deg = graph.degree(n)
        capacity[n] = (betweenness[n] ** 0.8) * (1 + alpha * np.log1p(deg)) + 0.001
    base_flood_depths = {n: graph.nodes[n].get('flood_depth_m', 0.0) for n in graph.nodes()}
    
    if not any('distance_from_epicenter' in data for _, data in graph.nodes(data=True)):
        print("Computing distance from epicenter...")
        if len(base_flood_depths) > 0:
            max_flood_node = max(base_flood_depths, key=base_flood_depths.get)
            max_depth = base_flood_depths.get(max_flood_node, 0)
            if max_depth > 0 and 'x' in graph.nodes[max_flood_node] and 'y' in graph.nodes[max_flood_node]:
                epi_x = graph.nodes[max_flood_node]['x']
                epi_y = graph.nodes[max_flood_node]['y']
                for n, d in graph.nodes(data=True):
                    if 'x' in d and 'y' in d:
                        d['distance_from_epicenter'] = np.sqrt((d['x'] - epi_x)**2 + (d['y'] - epi_y)**2)
                    else:
                        d['distance_from_epicenter'] = 0.0
                        
                distances = [d['distance_from_epicenter'] for n, d in graph.nodes(data=True)]
                max_d = max(distances) if distances and max(distances) > 0 else 1.0
                for n, d in graph.nodes(data=True):
                    d['distance_from_epicenter'] /= max_d
            else:
                for n, d in graph.nodes(data=True):
                    d['distance_from_epicenter'] = 0.0
        else:
            for n, d in graph.nodes(data=True):
                d['distance_from_epicenter'] = 0.0

    all_records = []
    
    for s in range(n_scenarios):
        print(f"Simulating scenario {s+1}/{n_scenarios}...")
        records = simulate_cascade(graph, s, n_timesteps, dependencies, betweenness, capacity, base_flood_depths, degree_cent)
        all_records.extend(records)
        
    df = pd.DataFrame(all_records)
    
    for col in ['betweenness_centrality', 'redistributed_load', 'overload_ratio']:
        if col in df.columns:
            # Shift slightly to handle exact zeroes
            df[f'{col}_log'] = np.log1p(df[col] * 100)
            
    print(f"Generated dataset with {len(df)} rows.")
    return df

if __name__ == '__main__':
    print("Loading graph...")
    with open('cascadewatch_graph.pkl', 'rb') as f:
        graph = pickle.load(f)
        
    for n, d in graph.nodes(data=True):
        if 'type' in d and 'node_type' not in d:
            d['node_type'] = d['type']
            
    df = generate_synthetic_dataset(graph, n_scenarios=5, n_timesteps=10, seed=42)
    df.to_csv('synthetic_cascade_dataset.csv', index=False)
    print("Saved to synthetic_cascade_dataset.csv")


Loading graph...
Building dependency map...
Computing centralities...
Computing distance from epicenter...
Simulating scenario 1/5...
Simulating scenario 2/5...
Simulating scenario 3/5...
Simulating scenario 4/5...
Simulating scenario 5/5...
Generated dataset with 41000 rows.
Saved to synthetic_cascade_dataset.csv


In [18]:
df.groupby('timestep')['target_failed'].mean()

timestep
0    0.088049
1    0.206098
2    0.392927
3    0.549756
4    0.725366
5    0.820732
6    0.885610
7    0.920488
8    0.945366
9    0.967561
Name: target_failed, dtype: float64

In [19]:
df.groupby(pd.cut(df['fragility_score'], bins=5))['target_failed'].mean()

/var/folders/mf/_7cg4fl53x78fg3zyp16prcm0000gn/T/ipykernel_38711/592956991.py:1: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  df.groupby(pd.cut(df['fragility_score'], bins=5))['target_failed'].mean()


fragility_score
(-0.000253, 0.201]    0.197087
(0.201, 0.4]          0.270270
(0.4, 0.6]            0.358491
(0.6, 0.8]            0.414634
(0.8, 1.0]            0.975203
Name: target_failed, dtype: float64

In [21]:
df.groupby(pd.cut(df['overload_ratio'], bins=5))['target_failed'].mean()

/var/folders/mf/_7cg4fl53x78fg3zyp16prcm0000gn/T/ipykernel_38711/4234801694.py:1: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  df.groupby(pd.cut(df['overload_ratio'], bins=5))['target_failed'].mean()


overload_ratio
(-0.478, 95.528]      0.649956
(95.528, 191.057]          NaN
(191.057, 286.585]    1.000000
(286.585, 382.113]         NaN
(382.113, 477.642]    1.000000
Name: target_failed, dtype: float64

In [16]:
df.groupby('cross_layer_dependency_stress')['target_failed'].mean()

cross_layer_dependency_stress
0.0    0.065392
0.1    0.169446
0.2    0.224784
0.3    0.249267
0.4    0.246883
0.5    0.298635
0.6    0.116279
0.6    0.305455
0.7    0.730000
0.7    0.357631
0.8    0.466151
0.9    0.441748
1.0    0.939912
Name: target_failed, dtype: float64

In [20]:
df['target_failed'].mean()

np.float64(0.6501951219512195)